In [ ]:
!pip install -q diffusers transformers accelerate peft xformers huggingface_hub Pillow matplotlib

import torch, gc, os
from pathlib import Path
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
from peft import PeftModel
import matplotlib.pyplot as plt

# ── AUTH ──────────────────────────────────────────────────────
HF_TOKEN = "YOUR_HF_TOKEN"   # ← paste your token here
os.environ["HF_TOKEN"] = HF_TOKEN

# ── USER CONFIG ───────────────────────────────────────────────
LOAD_MODE  = "adapter"

BASE_MODEL = "Laxhar/noobai-XL-Vpred-1.0"
ADAPTER_ID = "Arezki-Cherfouh/noobai-xl-lora"
MERGED_ID  = "Arezki-Cherfouh/noobai-xl-lora-merged"

NUM_IMAGES = 4
WIDTH      = 1024   # Colab T4 can handle 1024
HEIGHT     = 1024
STEPS      = 28
CFG        = 7.0
SEED       = 42

TRIGGER = "fcs"
PROMPT = (
    f"{TRIGGER}, 1boy, detailed_shading, sharp_linework, cel_shading, "
    "masterpiece, best_quality, absurdres"
)
NEGATIVE = (
    "lowres, bad_anatomy, bad_hands, missing_fingers, worst_quality, "
    "low_quality, jpeg_artifacts, blurry, 3d, photo, realistic"
)

OUT_DIR = Path("/content/outputs")
# ─────────────────────────────────────────────────────────────

OUT_DIR.mkdir(exist_ok=True)
gc.collect()
torch.cuda.empty_cache()

if LOAD_MODE == "adapter":
    print(f"Loading base + adapter ({ADAPTER_ID})...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        token=HF_TOKEN,
        use_safetensors=True,
        device_map="auto",
    )
    pipe.unet = PeftModel.from_pretrained(
        pipe.unet, ADAPTER_ID, token=HF_TOKEN, is_trainable=False)
    pipe.unet.eval()
    print("✅ Base + adapter loaded")

elif LOAD_MODE == "merged":
    print(f"Loading merged model ({MERGED_ID})...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MERGED_ID,
        torch_dtype=torch.float16,
        token=HF_TOKEN,
        use_safetensors=True,
        device_map="auto",
    )
    print("✅ Merged model loaded")

pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config,
    prediction_type="v_prediction",
    rescale_betas_zero_snr=True,
    timestep_spacing="trailing",
)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
print("✅ Scheduler ready")

print(f"\nGenerating {NUM_IMAGES} image(s)...")
results = []
for i in range(NUM_IMAGES):
    seed = (SEED + i) if SEED is not None else torch.randint(0, 2**32, (1,)).item()
    print(f"  [{i+1}/{NUM_IMAGES}] seed={seed}")
    with torch.no_grad():
        img = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE,
            num_inference_steps=STEPS,
            guidance_scale=CFG,
            width=WIDTH,
            height=HEIGHT,
            generator=torch.Generator("cpu").manual_seed(seed),
        ).images[0]
    img.save(OUT_DIR / f"output_{i+1}_seed{seed}.png")
    results.append((img, f"seed {seed}"))
    print(f"     ✅ saved")

cols = min(NUM_IMAGES, 4)
rows = (NUM_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = [axes] if NUM_IMAGES == 1 else (
    [ax for row in axes for ax in row] if rows > 1 else list(axes))
for idx, (img, label) in enumerate(results):
    axes[idx].imshow(img); axes[idx].set_title(label, fontsize=10); axes[idx].axis("off")
for idx in range(NUM_IMAGES, len(axes)):
    axes[idx].axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "output_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ All images saved to {OUT_DIR}")